# Machine Learning Foundations
## Model Evaluation: Train/Test Split · Cross-Validation · Overfitting & Underfitting

---

### What This Notebook Covers

Training a model is only half the job. The harder question is: **does it actually work on data it has never seen?**

This notebook teaches the three core concepts that separate models that generalize from models that merely memorize:

| Concept | The Question It Answers |
|---|---|
| **Train/Test Split** | How do we get an honest estimate of real-world performance? |
| **Cross-Validation** | How do we get a *reliable* estimate when data is limited? |
| **Overfitting / Underfitting** | Is the model learning the signal — or the noise? |

Throughout, we use a real dataset of **1,338 US health insurance policyholders** to predict annual medical charges. Every concept is demonstrated with working code on this data.

---

## Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, learning_curve, validation_curve
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

# ── Load & encode the insurance dataset ──
df = pd.read_csv('insurance.csv')
df['smoker_enc']  = (df['smoker'] == 'yes').astype(int)
df['sex_enc']     = (df['sex'] == 'male').astype(int)
df = pd.get_dummies(df, columns=['region'], drop_first=True)

FEATURES = ['age', 'sex_enc', 'bmi', 'children', 'smoker_enc',
            'region_northwest', 'region_southeast', 'region_southwest']
TARGET   = 'charges'

X = df[FEATURES].values.astype(float)
y = df[TARGET].values.astype(float)

print(f'Dataset: {X.shape[0]} patients × {X.shape[1]} features')
print(f'Target  — charges: min=${y.min():,.0f}  mean=${y.mean():,.0f}  max=${y.max():,.0f}')
df[['age','bmi','children','smoker_enc','charges']].describe().round(2)

In [ ]:
# ── Quick visual overview of the dataset ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Charges distribution
axes[0].hist(y, bins=45, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(y.mean(), color='crimson', linestyle='--', lw=2, label=f'Mean = ${y.mean():,.0f}')
axes[0].set_title('Distribution of Insurance Charges', fontsize=12)
axes[0].set_xlabel('Annual Charges ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Age vs Charges
c = df['smoker_enc'].map({1: 'crimson', 0: 'steelblue'})
axes[1].scatter(df['age'], y, c=c, alpha=0.3, s=12)
axes[1].set_title('Age vs Charges\n(red = smoker)', fontsize=12)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Charges ($)')

# BMI vs Charges
axes[2].scatter(df['bmi'], y, c=c, alpha=0.3, s=12)
axes[2].set_title('BMI vs Charges\n(red = smoker)', fontsize=12)
axes[2].set_xlabel('BMI')
axes[2].set_ylabel('Charges ($)')

plt.suptitle('Insurance Dataset Overview — 1,338 Policyholders', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 1 — Train / Test Split

### 1.1 The Fundamental Problem: Evaluating What the Model Has Never Seen

Imagine you teach a student for an exam using a set of practice problems. On exam day, you hand them the *exact same problems* they already practiced. Does a perfect score tell you they've learned the material? Absolutely not — they may have simply memorized the answers.

Machine learning models face the same trap. A model can always score perfectly on its **training data** by memorizing every example. That tells you nothing about whether it will work on new patients, new customers, or new data from next month.

The solution is straightforward: **hold out a portion of your data before training begins, and never let the model see it until you're ready for the final evaluation.** This held-out portion is the **test set**.

---

### 1.2 How the Split Works

You randomly shuffle your dataset and divide it into two non-overlapping subsets:

```
Full Dataset (1,338 patients)
│
├── Training Set (~80%) ──► Model learns from this
│   1,070 patients
│
└── Test Set (~20%) ───────► Model is evaluated on this (unseen)
    268 patients
```

**The cardinal rule:** the test set is locked away. You never use it to tune parameters, choose features, or make any decisions about the model. It is a one-time final exam.

**Typical split ratios:**
- **80/20** — most common; works well for datasets of 1,000+ examples
- **70/30** — gives a larger test set for more stable estimates
- **90/10** — when data is scarce and you need as much training data as possible

---

### 1.3 Why Random Shuffling Matters

If your data is ordered (e.g., sorted by age, or chronologically), taking the last 20% without shuffling would give you a biased test set. Always shuffle — unless you're dealing with time-series data, where temporal ordering must be preserved.

In [ ]:
# ────────────────────────────────────────────────────────────
# Demonstrating the danger of an unshuffled split
# ────────────────────────────────────────────────────────────

# The insurance dataset happens to be roughly sorted by some order.
# Let's show that naive splitting (no shuffle) can produce
# a biased test set with very different charge distributions.

split_idx = int(0.8 * len(y))

y_train_naive = y[:split_idx]
y_test_naive  = y[split_idx:]

# Now compare to a properly shuffled split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Naive split
axes[0].hist(y_train_naive, bins=35, alpha=0.6, color='steelblue', label=f'Train  mean=${y_train_naive.mean():,.0f}')
axes[0].hist(y_test_naive,  bins=35, alpha=0.6, color='crimson',   label=f'Test   mean=${y_test_naive.mean():,.0f}')
axes[0].set_title('❌  Naive Split (no shuffle)\nTrain and test have different distributions', fontsize=11)
axes[0].set_xlabel('Charges ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Shuffled split
axes[1].hist(y_train, bins=35, alpha=0.6, color='steelblue', label=f'Train  mean=${y_train.mean():,.0f}')
axes[1].hist(y_test,  bins=35, alpha=0.6, color='crimson',   label=f'Test   mean=${y_test.mean():,.0f}')
axes[1].set_title('✅  Shuffled Split (random_state=42)\nTrain and test distributions match', fontsize=11)
axes[1].set_xlabel('Charges ($)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Always Shuffle Before Splitting', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Naive split   — Train mean: ${y_train_naive.mean():,.0f}  |  Test mean: ${y_test_naive.mean():,.0f}')
print(f'Shuffled split — Train mean: ${y_train.mean():,.0f}  |  Test mean: ${y_test.mean():,.0f}')
print(f'\nShuffled split: {len(y_train)} training samples, {len(y_test)} test samples')

In [ ]:
# ────────────────────────────────────────────────────────────
# Train a Linear Regression model and evaluate properly
# ────────────────────────────────────────────────────────────

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit ONLY on training data
X_test_s  = scaler.transform(X_test)        # apply the same transform to test

model = LinearRegression()
model.fit(X_train_s, y_train)

# Predictions
y_pred_train = model.predict(X_train_s)
y_pred_test  = model.predict(X_test_s)

# Metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse  = np.sqrt(mean_squared_error(y_test,  y_pred_test))
train_r2   = r2_score(y_train, y_pred_train)
test_r2    = r2_score(y_test,  y_pred_test)

print('=== Linear Regression — Train vs Test Performance ===')
print(f'              Train        Test')
print(f'RMSE       ${train_rmse:>9,.0f}   ${test_rmse:>9,.0f}')
print(f'R²          {train_r2:>8.4f}    {test_r2:>8.4f}')
print()
print('Key insight:')
print('  Train R² tells us how well the model fit the training data.')
print('  Test R²  tells us how well it GENERALIZES to new, unseen data.')
print('  A large gap between the two signals overfitting (coming in Part 3).')

In [ ]:
# ────────────────────────────────────────────────────────────
# Critical warning: Data Leakage
# Showing what happens if you fit the scaler on ALL data
# ────────────────────────────────────────────────────────────
#
# DATA LEAKAGE occurs when information from the test set
# "leaks" into the training process, making test performance
# look artificially better than it really is.
#
# A common mistake: fitting the StandardScaler on the full dataset
# before splitting. The scaler then "knows" the test set's mean and std.

# ❌ Wrong approach
scaler_leaky = StandardScaler()
X_all_scaled_leaky = scaler_leaky.fit_transform(X)  # sees test data!
X_train_leaky = X_all_scaled_leaky[df.index.isin(range(len(X_train)))]

# ✅ Correct approach (what we already did)
# scaler.fit_transform(X_train)  → learns mean/std from train only
# scaler.transform(X_test)       → applies train's stats to test

print('Data Leakage — The Subtle Bug That Inflates Test Scores')
print('=' * 57)
print()
print('  ❌ Leaky pipeline:  fit scaler on ALL data → split → train/test')
print('  ✅ Clean pipeline:  split first → fit scaler on TRAIN only')
print()
print('The fix in sklearn: use a Pipeline object so all transformations')
print('are automatically fitted on training data only during CV.')
print()

# Show the clean pipeline pattern
clean_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])
clean_pipeline.fit(X_train, y_train)
pipeline_r2 = r2_score(y_test, clean_pipeline.predict(X_test))
print(f'Pipeline test R² = {pipeline_r2:.4f}  (should match our manual result: {test_r2:.4f})')

### 1.4 The Variance Problem — A Single Split is Fragile

A train/test split gives you **one** estimate of performance. But that estimate depends heavily on which 20% happened to end up in the test set. A lucky draw might give you R² = 0.78; an unlucky one might give 0.71. This variance is a problem when:

- You're comparing two competing models and the difference is small
- Your dataset is small (under ~1,000 rows)
- You need to tune hyperparameters without burning your test set

The next section introduces **cross-validation**, which solves this.

In [ ]:
# ────────────────────────────────────────────────────────────
# Demonstrate split variance: run 20 different random splits
# and observe how much the test R² fluctuates
# ────────────────────────────────────────────────────────────

r2_scores_by_seed = []

for seed in range(50):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    pipe = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())])
    pipe.fit(Xtr, ytr)
    r2_scores_by_seed.append(r2_score(yte, pipe.predict(Xte)))

r2_arr = np.array(r2_scores_by_seed)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(r2_arr, marker='o', markersize=5, color='steelblue', linewidth=1.5, label='Test R² per split')
ax.axhline(r2_arr.mean(), color='crimson', linestyle='--', lw=2,
           label=f'Mean R² = {r2_arr.mean():.4f}')
ax.fill_between(range(50),
                r2_arr.mean() - r2_arr.std(),
                r2_arr.mean() + r2_arr.std(),
                alpha=0.15, color='crimson', label=f'±1 std = ±{r2_arr.std():.4f}')
ax.set_title('R² across 50 Different Random Train/Test Splits\n'
             'Same model, same data — performance fluctuates based on which samples land in the test set',
             fontsize=12)
ax.set_xlabel('Random Seed (split index)')
ax.set_ylabel('Test R²')
ax.legend()
plt.tight_layout()
plt.show()

print(f'R² range: {r2_arr.min():.4f} → {r2_arr.max():.4f}')
print(f'Spread (max - min) = {r2_arr.max() - r2_arr.min():.4f}')
print(f'\nThis {r2_arr.max() - r2_arr.min():.3f} spread means a single split can make a mediocre')
print('model look good or a good model look mediocre — purely by chance.')

---
## Part 2 — Cross-Validation

### 2.1 The Core Idea: Use Every Example for Both Training and Testing

Cross-validation (CV) solves the variance problem by asking: instead of one split, what if we repeated the split *k* times, each time using a different portion of data as the test set?

---

### 2.2 K-Fold Cross-Validation

**K-Fold CV** is the most common form. The procedure:

1. Divide the dataset into **k equal folds** (partitions)
2. For each fold $i = 1, 2, ..., k$:
   - Use fold $i$ as the **validation set**
   - Train on the remaining $k-1$ folds
   - Record the validation score
3. The final CV score is the **average** of all $k$ validation scores

```
k = 5   (5-Fold CV)

Fold 1: [TEST ][train][train][train][train]  → Score₁
Fold 2: [train][TEST ][train][train][train]  → Score₂
Fold 3: [train][train][TEST ][train][train]  → Score₃
Fold 4: [train][train][train][TEST ][train]  → Score₄
Fold 5: [train][train][train][train][TEST ]  → Score₅
                                               ──────────────────────
                                        CV Score = mean(Score₁..₅)
```

Every example is used exactly **once** for validation and **k-1 times** for training.

---

### 2.3 Choosing k

| k | Behaviour | Best Used When |
|---|---|---|
| **k = 5** | Fast; moderate variance | Large datasets (1,000+) |
| **k = 10** | Standard choice; lower variance | General use |
| **k = m** (Leave-One-Out) | Lowest bias; very high variance + slow | Tiny datasets (<100) |

For our insurance dataset (1,338 rows), **5-fold or 10-fold** are both appropriate.

---

### 2.4 Stratified K-Fold

For **classification** problems, regular K-Fold can accidentally put all rare-class examples into one fold. **Stratified K-Fold** ensures each fold has the same class proportion as the full dataset. For regression, regular K-Fold is standard.

In [ ]:
# ────────────────────────────────────────────────────────────
# Implement K-Fold CV manually so the mechanics are transparent
# ────────────────────────────────────────────────────────────

def manual_kfold_cv(X, y, k=5):
    """
    Manual k-fold cross-validation for a linear regression pipeline.
    Returns per-fold RMSE and R² scores.
    """
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_rmse = []
    fold_r2   = []
    fold_sizes = []

    print(f'{k}-Fold Cross-Validation — Linear Regression on Insurance Charges')
    print('─' * 70)
    print(f'{"Fold":>5}  {"Train N":>8}  {"Val N":>6}  {"Val RMSE":>12}  {"Val R²":>8}')
    print('─' * 70)

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Pipeline: scale then fit — no leakage
        pipe = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())])
        pipe.fit(X_tr, y_tr)
        y_hat = pipe.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_hat))
        r2   = r2_score(y_val, y_hat)

        fold_rmse.append(rmse)
        fold_r2.append(r2)
        fold_sizes.append(len(val_idx))

        print(f'{fold_idx:>5}  {len(train_idx):>8}  {len(val_idx):>6}  ${rmse:>11,.0f}  {r2:>8.4f}')

    print('─' * 70)
    print(f'{"MEAN":>5}  {"":>8}  {"":>6}  ${np.mean(fold_rmse):>11,.0f}  {np.mean(fold_r2):>8.4f}')
    print(f'{"STD":>5}  {"":>8}  {"":>6}  ${np.std(fold_rmse):>11,.0f}  {np.std(fold_r2):>8.4f}')
    return np.array(fold_rmse), np.array(fold_r2)

rmse_folds, r2_folds = manual_kfold_cv(X, y, k=5)

In [ ]:
# ────────────────────────────────────────────────────────────
# CV vs single-split: a direct comparison
# ────────────────────────────────────────────────────────────

# sklearn shorthand (equivalent to our manual version)
pipe = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())])
cv_r2_scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
cv_r2_scores_10 = cross_val_score(pipe, X, y, cv=10, scoring='r2')

# Compare stability: CV vs 50 random single splits (from Part 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: variability comparison
data_to_plot = [r2_arr, cv_r2_scores, cv_r2_scores_10]
labels       = ['50 Random\nSingle Splits', '5-Fold CV\n(5 scores)', '10-Fold CV\n(10 scores)']
colors_bp    = ['#d95f02', '#1b9e77', '#7570b3']

bp = axes[0].boxplot(data_to_plot, patch_artist=True, labels=labels,
                     medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[0].set_title('Stability of R² Estimates\nCross-validation reduces variance in performance estimates', fontsize=11)
axes[0].set_ylabel('R²')
axes[0].set_ylim(0.6, 0.85)

# Right: per-fold breakdown
fold_nums = np.arange(1, 6)
axes[1].bar(fold_nums - 0.2, r2_folds, width=0.4, color='steelblue', alpha=0.8, label='Per-fold R²')
axes[1].axhline(r2_folds.mean(), color='crimson', linestyle='--', lw=2,
                label=f'CV Mean = {r2_folds.mean():.4f}')
axes[1].fill_between([-0.5, 5.5],
                     r2_folds.mean() - r2_folds.std(),
                     r2_folds.mean() + r2_folds.std(),
                     alpha=0.1, color='crimson', label=f'±1 std = ±{r2_folds.std():.4f}')
axes[1].set_title('R² per Fold — 5-Fold CV', fontsize=11)
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Validation R²')
axes[1].set_xticks(fold_nums)
axes[1].legend()

plt.suptitle('Cross-Validation gives a More Reliable Performance Estimate', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Single-split std:  {r2_arr.std():.4f}')
print(f'5-Fold CV std:     {cv_r2_scores.std():.4f}')
print(f'10-Fold CV std:    {cv_r2_scores_10.std():.4f}')

In [ ]:
# ────────────────────────────────────────────────────────────
# Practical use of CV: model selection
# Which model family is best for predicting insurance charges?
# Use CV to compare fairly — no test set touched yet
# ────────────────────────────────────────────────────────────

models = {
    'Baseline (mean)':    Pipeline([('sc', StandardScaler()), ('m', DummyRegressor(strategy='mean'))]),
    'Linear Regression':  Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Ridge (α=10)':       Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=10))]),
    'Poly deg-2':         Pipeline([('sc', StandardScaler()), ('pf', PolynomialFeatures(2, include_bias=False)), ('m', LinearRegression())]),
    'Decision Tree d=3':  Pipeline([('sc', StandardScaler()), ('m', DecisionTreeRegressor(max_depth=3, random_state=42))]),
    'Decision Tree d=10': Pipeline([('sc', StandardScaler()), ('m', DecisionTreeRegressor(max_depth=10, random_state=42))]),
}

print('Model Comparison via 5-Fold Cross-Validation')
print('=' * 60)
print(f'{"Model":25s}  {"CV R² mean":>12}  {"CV R² std":>10}  {"CV RMSE":>10}')
print('-' * 60)

cv_results = {}
for name, pipe in models.items():
    r2_cv   = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    rmse_cv = np.sqrt(-cross_val_score(pipe, X, y, cv=5, scoring='neg_mean_squared_error'))
    cv_results[name] = {'r2_mean': r2_cv.mean(), 'r2_std': r2_cv.std(), 'rmse_mean': rmse_cv.mean()}
    print(f'{name:25s}  {r2_cv.mean():>12.4f}  {r2_cv.std():>10.4f}  ${rmse_cv.mean():>9,.0f}')

print()
best = max(cv_results, key=lambda k: cv_results[k]['r2_mean'])
print(f'Best model by CV R²: {best}')
print(f'Note: Decision Tree depth=10 may look strong but watch it overfit in Part 3!')

In [ ]:
# ────────────────────────────────────────────────────────────
# The right workflow: CV for model selection, test set for final report
# ────────────────────────────────────────────────────────────
#
# Step 1: Split off a test set — lock it away
# Step 2: Use CV on training data to compare models / tune hyperparameters
# Step 3: Re-train the best model on ALL training data
# Step 4: Evaluate ONCE on the test set — this is your published result

X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Correct ML Workflow')
print('─' * 55)
print(f'Step 1: Hold out test set ({len(y_test_final)} samples) — LOCKED')

# Step 2: CV on training data only
best_pipe = Pipeline([('sc', StandardScaler()), ('m', LinearRegression())])
cv_scores = cross_val_score(best_pipe, X_train_full, y_train_full, cv=5, scoring='r2')
print(f'Step 2: 5-Fold CV on training data → R² = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Step 3: Re-train on ALL training data
best_pipe.fit(X_train_full, y_train_full)
print(f'Step 3: Retrain on full training set ({len(y_train_full)} samples)')

# Step 4: One-time final evaluation
final_r2 = r2_score(y_test_final, best_pipe.predict(X_test_final))
print(f'Step 4: Final test R² = {final_r2:.4f}  ← this is the honest reported result')

---
## Part 3 — Overfitting and Underfitting

### 3.1 The Bias-Variance Trade-off

Every model makes two kinds of errors:

- **Bias** — error from wrong assumptions. A model that assumes a linear relationship when the truth is curved will consistently make the same types of errors, regardless of the training data. High bias → **underfitting**.

- **Variance** — error from sensitivity to small fluctuations in training data. A model that is too complex will fit the noise in one training set and produce completely different predictions if retrained on a slightly different sample. High variance → **overfitting**.

The total expected error decomposes as:

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

You cannot eliminate irreducible noise — it's the inherent randomness in the data. But the goal of model selection and regularization is to find the **sweet spot** where bias² + variance is minimized.

---

### 3.2 Underfitting — Too Simple

An **underfit** model is too simple to capture the true patterns in the data. It performs poorly on both training *and* test data. Signs:

- Training R² is low (the model doesn't even fit training data well)
- Training error ≈ test error (both are high)
- Adding more training data doesn't help much

**Fix:** Use a more complex model, add features, or reduce regularization.

---

### 3.3 Overfitting — Too Complex

An **overfit** model has memorized the training data — it has learned the noise along with the signal. It performs excellently on training data but fails on new data. Signs:

- Training R² is very high (near 1.0)
- Test R² is significantly lower
- **Large gap** between training and test scores
- Performance gets worse as model complexity increases

**Fix:** Collect more data, reduce model complexity, add regularization (Ridge/Lasso), or use dropout (neural networks).

---

### 3.4 The Model Complexity Spectrum

```
 Complexity ──────────────────────────────────────────────►

 [Underfit]          [Just Right]           [Overfit]
 High Bias           Low Bias               Low Bias
 Low Variance        Low Variance           High Variance
 Train R² low        Train R² high          Train R² ≈ 1.0
 Test R² low         Test R² high           Test R² drops
 Gap: small          Gap: small             Gap: LARGE
```

In [ ]:
# ────────────────────────────────────────────────────────────
# Visualize overfitting with polynomial regression
# Using a single feature (age) to keep the plots interpretable
# ────────────────────────────────────────────────────────────

# Use only non-smokers for a cleaner curve (smokers form a separate cluster)
mask      = df['smoker_enc'] == 0
X_age_ns  = df.loc[mask, 'age'].values.reshape(-1, 1)
y_ns      = df.loc[mask, 'charges'].values

# Train on a deliberately small subset to make overfitting obvious
Xtr_a, Xte_a, ytr_a, yte_a = train_test_split(X_age_ns, y_ns, test_size=0.85, random_state=7)

degrees     = [1, 2, 4, 9, 15]
x_plot      = np.linspace(X_age_ns.min(), X_age_ns.max(), 300).reshape(-1, 1)

fig, axes   = plt.subplots(1, len(degrees), figsize=(20, 5), sharey=True)

for ax, deg in zip(axes, degrees):
    pipe_poly = Pipeline([
        ('sc',  StandardScaler()),
        ('pf',  PolynomialFeatures(deg, include_bias=False)),
        ('lr',  LinearRegression())
    ])
    pipe_poly.fit(Xtr_a, ytr_a)

    train_r2 = r2_score(ytr_a, pipe_poly.predict(Xtr_a))
    test_r2  = r2_score(yte_a, pipe_poly.predict(Xte_a))

    # Scatter: training (blue dots) + test (grey dots)
    ax.scatter(Xte_a, yte_a, color='silver',   alpha=0.3, s=8,  label='Test data')
    ax.scatter(Xtr_a, ytr_a, color='steelblue', alpha=0.9, s=25, label='Train data', zorder=3)
    ax.plot(x_plot, pipe_poly.predict(x_plot), color='crimson', lw=2)

    gap = train_r2 - test_r2
    if deg == 1:
        verdict = 'UNDERFIT'
        vcolor  = 'darkorange'
    elif deg in [2, 4]:
        verdict = 'GOOD FIT'
        vcolor  = 'green'
    else:
        verdict = 'OVERFIT'
        vcolor  = 'crimson'

    ax.set_title(f'Degree {deg}\n'
                 f'Train R²={train_r2:.2f}  Test R²={test_r2:.2f}\n'
                 f'Gap = {gap:.2f}   [{verdict}]',
                 fontsize=9.5,
                 color=vcolor)
    ax.set_xlabel('Age')
    ax.set_ylim(-5000, 30000)

axes[0].set_ylabel('Charges ($)')
axes[0].legend(fontsize=8)

plt.suptitle('Polynomial Degree vs Fit Quality — Non-Smoker Insurance Charges\n'
             'As complexity increases: underfitting → good fit → overfitting',
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# ────────────────────────────────────────────────────────────
# The classic Train/Test error curve vs Model Complexity
# ────────────────────────────────────────────────────────────

degrees_full = list(range(1, 13))
train_r2_all, test_r2_all = [], []

# Use full dataset for this curve (all features → polynomial on scaled X)
Xtr_f, Xte_f, ytr_f, yte_f = train_test_split(X, y, test_size=0.2, random_state=42)

for deg in degrees_full:
    pipe_d = Pipeline([
        ('sc', StandardScaler()),
        ('pf', PolynomialFeatures(deg, include_bias=False)),
        ('lr', LinearRegression())
    ])
    pipe_d.fit(Xtr_f, ytr_f)
    train_r2_all.append(r2_score(ytr_f, pipe_d.predict(Xtr_f)))
    test_r2_all.append(r2_score(yte_f, pipe_d.predict(Xte_f)))

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(degrees_full, train_r2_all, 'o-', color='steelblue', lw=2.5, label='Train R²')
ax.plot(degrees_full, test_r2_all,  's-', color='crimson',   lw=2.5, label='Test R²')
ax.fill_between(degrees_full, train_r2_all, test_r2_all,
                alpha=0.12, color='crimson', label='Generalization gap')

ax.axvline(2, color='green', linestyle=':', lw=2, label='Sweet spot ≈ deg 2')

ax.annotate('Underfitting region\n(high bias)', xy=(1, train_r2_all[0]),
            xytext=(1.3, 0.55), fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))
ax.annotate('Overfitting region\n(high variance)', xy=(9, test_r2_all[8]),
            xytext=(7.5, 0.58), fontsize=9, color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson'))

ax.set_title('Model Complexity Curve — Train R² vs Test R²\n'
             'Training score always rises; test score peaks then falls', fontsize=12)
ax.set_xlabel('Polynomial Degree (proxy for model complexity)')
ax.set_ylabel('R²')
ax.legend()
ax.set_xticks(degrees_full)
plt.tight_layout()
plt.show()

best_deg = degrees_full[np.argmax(test_r2_all)]
print(f'Best test R² at degree {best_deg}: {max(test_r2_all):.4f}')
print(f'Train R² at degree {len(degrees_full)}: {train_r2_all[-1]:.4f} — nearly perfect on training data')
print(f'Test  R² at degree {len(degrees_full)}: {test_r2_all[-1]:.4f}  — but generalizes poorly')

In [ ]:
# ────────────────────────────────────────────────────────────
# Learning Curves — Diagnosing bias vs variance from data size
# ────────────────────────────────────────────────────────────
#
# A learning curve plots training and validation score
# as a function of training set SIZE.
#
# Pattern for HIGH BIAS (underfitting):
#   - Both curves plateau at a LOW score
#   - Adding more data doesn't help
#
# Pattern for HIGH VARIANCE (overfitting):
#   - Training score is high, validation score is much lower
#   - Adding more data HELPS — the gap closes

def plot_learning_curve(model, X, y, title, ax, cv=5):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        train_sizes=np.linspace(0.05, 1.0, 20),
        cv=cv, scoring='r2', n_jobs=-1
    )
    tr_mean = train_scores.mean(axis=1)
    tr_std  = train_scores.std(axis=1)
    vl_mean = val_scores.mean(axis=1)
    vl_std  = val_scores.std(axis=1)

    ax.plot(train_sizes, tr_mean, 'o-', color='steelblue', lw=2, label='Training R²')
    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color='steelblue')
    ax.plot(train_sizes, vl_mean, 's-', color='crimson', lw=2, label='CV Validation R²')
    ax.fill_between(train_sizes, vl_mean - vl_std, vl_mean + vl_std, alpha=0.15, color='crimson')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('R²')
    ax.legend(fontsize=9)
    ax.set_ylim(-0.1, 1.05)


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Underfit model: linear regression on only 1 feature
underfit_model = Pipeline([
    ('sc', StandardScaler()),
    ('lr', LinearRegression())
])
plot_learning_curve(underfit_model,
                    df[['age']].values, y,
                    'Underfit Model\n(Linear, 1 feature: age only)',
                    axes[0])

# Good model: linear regression on all features
good_model = Pipeline([
    ('sc', StandardScaler()),
    ('lr', LinearRegression())
])
plot_learning_curve(good_model, X, y,
                    'Good Model\n(Linear Regression, all features)',
                    axes[1])

# Overfit model: high-depth decision tree
overfit_model = Pipeline([
    ('sc', StandardScaler()),
    ('dt', DecisionTreeRegressor(max_depth=15, random_state=42))
])
plot_learning_curve(overfit_model, X, y,
                    'Overfit Model\n(Decision Tree, max_depth=15)',
                    axes[2])

plt.suptitle('Learning Curves — Diagnosing Underfitting vs Overfitting from Data Size',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Reading learning curves:')
print('  Underfit : Both curves plateau LOW → model too simple, more data won\'t help')
print('  Good fit : Curves converge to a HIGH score → healthy model')
print('  Overfit  : Large gap, train curve high → more data would NARROW the gap')

In [ ]:
# ────────────────────────────────────────────────────────────
# Validation Curve — tune a single hyperparameter
# How does Decision Tree max_depth affect train vs CV performance?
# ────────────────────────────────────────────────────────────

depths = np.arange(1, 21)

dt_pipe = Pipeline([
    ('sc', StandardScaler()),
    ('dt', DecisionTreeRegressor(random_state=42))
])

train_scores_vc, val_scores_vc = validation_curve(
    dt_pipe, X, y,
    param_name='dt__max_depth',
    param_range=depths,
    cv=5, scoring='r2'
)

tr_m = train_scores_vc.mean(axis=1)
vl_m = val_scores_vc.mean(axis=1)
tr_s = train_scores_vc.std(axis=1)
vl_s = val_scores_vc.std(axis=1)

best_depth = depths[np.argmax(vl_m)]

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(depths, tr_m, 'o-', color='steelblue', lw=2.5, label='Train R²')
ax.fill_between(depths, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color='steelblue')
ax.plot(depths, vl_m, 's-', color='crimson', lw=2.5, label='CV Validation R²')
ax.fill_between(depths, vl_m - vl_s, vl_m + vl_s, alpha=0.15, color='crimson')

ax.axvline(best_depth, color='green', linestyle='--', lw=2,
           label=f'Best depth = {best_depth}  (CV R²={vl_m[best_depth-1]:.4f})')

ax.set_title('Validation Curve — Decision Tree max_depth on Insurance Charges\n'
             'Shallow = underfit → optimal depth → deep = overfit',
             fontsize=12)
ax.set_xlabel('max_depth (model complexity →)')
ax.set_ylabel('R²')
ax.set_xticks(depths)
ax.legend()

# Annotate the three regimes
ax.annotate('Underfit', xy=(2, vl_m[1]), xytext=(2.5, 0.45),
            fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))
ax.annotate('Overfit', xy=(15, vl_m[14]), xytext=(14, 0.58),
            fontsize=9, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred'))

plt.tight_layout()
plt.show()

print(f'Optimal max_depth = {best_depth}  →  CV R² = {vl_m[best_depth-1]:.4f}')
print(f'At max_depth=20:  Train R²={tr_m[-1]:.4f}  |  CV R²={vl_m[-1]:.4f}  |  Gap={tr_m[-1]-vl_m[-1]:.4f}')

In [ ]:
# ────────────────────────────────────────────────────────────
# Fixing Overfitting with Regularization — Ridge Regression
# ────────────────────────────────────────────────────────────
#
# Regularization adds a penalty term to the cost function
# that discourages large coefficients:
#
#   Ridge cost = MSE + α * sum(θ²)
#
# Larger α → stronger shrinkage → simpler model
# α = 0     → plain linear regression

alphas = np.logspace(-3, 5, 60)   # 0.001 to 100,000

# High-degree polynomial to deliberately overfit first
cv_ridge_scores = []
for alpha in alphas:
    ridge_pipe = Pipeline([
        ('sc', StandardScaler()),
        ('pf', PolynomialFeatures(3, include_bias=False)),
        ('r',  Ridge(alpha=alpha))
    ])
    scores = cross_val_score(ridge_pipe, X, y, cv=5, scoring='r2')
    cv_ridge_scores.append(scores.mean())

cv_ridge_scores = np.array(cv_ridge_scores)
best_alpha      = alphas[np.argmax(cv_ridge_scores)]

fig, ax = plt.subplots(figsize=(11, 5))
ax.semilogx(alphas, cv_ridge_scores, 'o-', color='steelblue', lw=2, markersize=4)
ax.axvline(best_alpha, color='crimson', linestyle='--', lw=2,
           label=f'Best α = {best_alpha:.1f}  (CV R²={max(cv_ridge_scores):.4f})')
ax.set_title('Ridge Regularization — CV R² vs Penalty Strength (α)\n'
             'Too little regularization → overfit | Too much → underfit',
             fontsize=12)
ax.set_xlabel('Ridge α (regularization strength, log scale →)')
ax.set_ylabel('5-Fold CV R²')
ax.legend()

ax.annotate('Underfitting\n(α too large)',
            xy=(alphas[-5], cv_ridge_scores[-5]),
            xytext=(1e3, cv_ridge_scores.min() + 0.05),
            fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))
ax.annotate('Overfitting\n(α too small)',
            xy=(alphas[2], cv_ridge_scores[2]),
            xytext=(0.005, cv_ridge_scores[2] - 0.06),
            fontsize=9, color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson'))

plt.tight_layout()
plt.show()

print(f'Best Ridge α = {best_alpha:.2f}')
print(f'This is found entirely through cross-validation — the test set was never touched.')

---
## Part 4 — Putting It All Together

### The Complete, Correct ML Workflow

Everything we have covered feeds into a single disciplined workflow. Here it is applied end-to-end on the insurance dataset.

In [ ]:
# ────────────────────────────────────────────────────────────
# Complete end-to-end pipeline on insurance charges
# ────────────────────────────────────────────────────────────

print('STEP 1 — Hold out the test set (20%) — this is LOCKED until the very end')
X_dev, X_hold, y_dev, y_hold = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'  Development set: {len(y_dev)} samples')
print(f'  Held-out test  : {len(y_hold)} samples  ← not touched until Step 4')

print()
print('STEP 2 — Compare candidate models using 5-Fold CV on development set only')

candidates = {
    'Linear Regression':     Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Ridge (best α)':        Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=best_alpha))]),
    'Poly-2 + Ridge':        Pipeline([('sc', StandardScaler()), ('pf', PolynomialFeatures(2, include_bias=False)), ('m', Ridge(alpha=best_alpha))]),
    'Decision Tree d=best':  Pipeline([('sc', StandardScaler()), ('m', DecisionTreeRegressor(max_depth=best_depth, random_state=42))]),
}

best_score = -np.inf
best_name  = None
best_pipe_final = None

print(f'  {"Model":30s}  {"CV R² mean":>12}  {"CV R² std":>10}')
print('  ' + '-' * 57)
for name, pipe_c in candidates.items():
    cv_sc = cross_val_score(pipe_c, X_dev, y_dev, cv=5, scoring='r2')
    print(f'  {name:30s}  {cv_sc.mean():>12.4f}  {cv_sc.std():>10.4f}')
    if cv_sc.mean() > best_score:
        best_score = cv_sc.mean()
        best_name  = name
        best_pipe_final = pipe_c

print(f'\n  Winner: {best_name}  (CV R² = {best_score:.4f})')

print()
print('STEP 3 — Retrain winning model on the FULL development set')
best_pipe_final.fit(X_dev, y_dev)
print(f'  Trained on {len(y_dev)} samples.')

print()
print('STEP 4 — ONE-TIME evaluation on the held-out test set')
y_final_pred = best_pipe_final.predict(X_hold)
final_r2   = r2_score(y_hold, y_final_pred)
final_rmse = np.sqrt(mean_squared_error(y_hold, y_final_pred))
final_mae  = mean_absolute_error(y_hold, y_final_pred)
print(f'  Model   : {best_name}')
print(f'  Test R² : {final_r2:.4f}')
print(f'  Test RMSE: ${final_rmse:,.0f}')
print(f'  Test MAE : ${final_mae:,.0f}')
print(f'  CV estimate was {best_score:.4f} — actual test is {final_r2:.4f}  (gap = {abs(best_score - final_r2):.4f})')

In [ ]:
# ────────────────────────────────────────────────────────────
# Final summary visualisation
# ────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

# ── Panel 1: Predicted vs Actual ──
ax1 = fig.add_subplot(gs[0])
ax1.scatter(y_hold, y_final_pred, alpha=0.35, s=12, color='steelblue')
mn, mx = y_hold.min(), y_hold.max()
ax1.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect prediction')
ax1.set_xlabel('Actual Charges ($)')
ax1.set_ylabel('Predicted Charges ($)')
ax1.set_title(f'Predicted vs Actual\nTest R² = {final_r2:.4f}', fontsize=11)
ax1.legend(fontsize=8)

# ── Panel 2: Residuals ──
ax2 = fig.add_subplot(gs[1])
residuals = y_hold - y_final_pred
ax2.hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(0,               color='crimson', lw=2, linestyle='--', label='Zero error')
ax2.axvline(residuals.mean(), color='orange',  lw=2, linestyle='--',
            label=f'Mean = ${residuals.mean():,.0f}')
ax2.set_title('Residuals Distribution\n(Should be centred at 0)', fontsize=11)
ax2.set_xlabel('Actual − Predicted ($)')
ax2.set_ylabel('Count')
ax2.legend(fontsize=8)

# ── Panel 3: Concept recap bar chart ──
ax3 = fig.add_subplot(gs[2])
concept_data = {
    'Train R²\n(all features,\nLin. Reg.)':     train_r2,
    'Single-split\nTest R²':                     test_r2,
    '5-Fold\nCV R²':                             r2_folds.mean(),
    'Final\nTest R²':                            final_r2,
}
bars = ax3.bar(concept_data.keys(), concept_data.values(),
               color=['steelblue', 'darkorange', 'green', 'crimson'], alpha=0.8, edgecolor='white')
for bar, val in zip(bars, concept_data.values()):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
ax3.set_ylim(0, 1.0)
ax3.set_ylabel('R²')
ax3.set_title('R² Estimates by Evaluation Method', fontsize=11)

plt.suptitle(f'End-to-End Results — Best Model: {best_name}', fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

### Train / Test Split
Hold out a portion of data **before** training so you have an honest, unseen benchmark. Always shuffle. Never let information from the test set influence any preprocessing, feature selection, or model choice — that is **data leakage**. A single split is easy but fragile: the score depends on which samples happened to land in each partition.

### Cross-Validation
K-Fold CV solves the instability of a single split by rotating the validation window across the full dataset. Every example serves as both a training example and a validation example exactly once. Use 5-fold or 10-fold CV for model comparison and hyperparameter tuning — **only use the test set once, at the very end**.

### Overfitting and Underfitting
- **Underfitting (high bias):** model too simple → poor performance everywhere → fix with more features or higher complexity
- **Overfitting (high variance):** model too complex → great on training data, poor on test → fix with regularization, more data, or reduced complexity
- **Diagnosis tools:** the train/test gap, learning curves, and validation curves all tell you where on the bias-variance spectrum your model sits
- **The right complexity** is found with cross-validation — never by peeking at the test set

---
### The Golden Workflow

```
1. Split → hold out test set (locked)  
2. CV on train → compare models, tune hyperparameters  
3. Retrain winner on full train set  
4. Evaluate ONCE on test set → report this number
```

---
*Dataset: Medical Cost Personal Datasets — 1,338 US insurance policyholders*